In [42]:
import torch 
from torch import nn,optim
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt 
import pandas as pd
from transformers import BertTokenizer, BertModel
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report
from transformers import RobertaModel
from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_linear_schedule_with_warmup

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else"cpu")

In [44]:
df_fake = pd.read_csv(r"D:\Machine learning\deep learning\Fake_News\News _dataset\Fake.csv") 
df_fake['label'] = 1
df_fake

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1
...,...,...,...,...,...
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",1
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",1
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",1
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",1


In [45]:
df_true = pd.read_csv(r"D:\Machine learning\deep learning\Fake_News\News _dataset\True.csv")
df_true['label']=0
df_true

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",0
...,...,...,...,...,...
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",0
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",0
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",0
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",0


In [46]:
print(df_true.duplicated().sum())
print(df_fake.duplicated().sum())

206
3


In [47]:
df_fake['text'].iloc[2]

'On Friday, it was revealed that former Milwaukee Sheriff David Clarke, who was being considered for Homeland Security Secretary in Donald Trump s administration, has an email scandal of his own.In January, there was a brief run-in on a plane between Clarke and fellow passenger Dan Black, who he later had detained by the police for no reason whatsoever, except that maybe his feelings were hurt. Clarke messaged the police to stop Black after he deplaned, and now, a search warrant has been executed by the FBI to see the exchanges.Clarke is calling it fake news even though copies of the search warrant are on the Internet. I am UNINTIMIDATED by lib media attempts to smear and discredit me with their FAKE NEWS reports designed to silence me,  the former sheriff tweeted.  I will continue to poke them in the eye with a sharp stick and bitch slap these scum bags til they get it. I have been attacked by better people than them #MAGA I am UNINTIMIDATED by lib media attempts to smear and discredi

In [48]:
df_true['text'].iloc[2]

'WASHINGTON (Reuters) - The special counsel investigation of links between Russia and President Trump’s 2016 election campaign should continue without interference in 2018, despite calls from some Trump administration allies and Republican lawmakers to shut it down, a prominent Republican senator said on Sunday. Lindsey Graham, who serves on the Senate armed forces and judiciary committees, said Department of Justice Special Counsel Robert Mueller needs to carry on with his Russia investigation without political interference. “This investigation will go forward. It will be an investigation conducted without political influence,” Graham said on CBS’s Face the Nation news program. “And we all need to let Mr. Mueller do his job. I think he’s the right guy at the right time.”  The question of how Russia may have interfered in the election, and how Trump’s campaign may have had links with or co-ordinated any such effort, has loomed over the White House since Trump took office in January. It

In [49]:
df_fake.drop_duplicates().reset_index(drop=True)
df_true.drop_duplicates().reset_index(drop=True)

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",0
...,...,...,...,...,...
21206,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",0
21207,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",0
21208,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",0
21209,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",0


In [50]:
print(df_fake.shape)
print(df_true.shape)

(23481, 5)
(21417, 5)


In [51]:
df = pd.concat([df_fake,df_true],axis=0).reset_index(drop=True)
df

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1
...,...,...,...,...,...
44893,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",0
44894,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",0
44895,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",0
44896,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",0


In [52]:
df.shape

(44898, 5)

In [53]:
df['content'] = df['title']+df['text']+df['subject']
df = df.drop(columns=['title','text','subject','date'])

In [54]:
# Shuffle the entire dataset randomly
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


In [55]:
df['label'].value_counts()

label
1    23481
0    21417
Name: count, dtype: int64

In [56]:



import re, nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Shiwan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Shiwan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Shiwan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [57]:
df['cleaned_text'] = df['content'].apply(clean_text)

In [58]:
df = df.drop(columns=['content'],axis=1)

In [59]:
X= df['cleaned_text']
y= df['label']

In [60]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [61]:
list(X_train.values)

['zimbabwe mnangagwa arrives home sworn president fridayharare reuters zimbabwe former vice president emmerson mnangagwa arrived back country wednesday two day due sworn president replace robert mugabe ruling party zanu pf official larry mavhima said mugabe resigned zimbabwe president tuesday week army former political ally moved end four decade rule man feted independence hero became feared despot worldnews',
 'black chicago teen call cop time help execute video people always wonder black people reluctant call police help yet another example long line demonstrates wary law enforcement even victim year old chicago resident quintonio legrier called three time emergency dispatcher hang police arrived happens many case fatally shot people charged serving protecting root according abc news audio year old call made dec released late monday chicago independent police review authority investigating legrier death death neighbor bettie jones jones also shot killed police opened apartment buildi

In [62]:
import random

def denormalize_text(cleaned_text):
    """
    Attempts to reverse the preprocessing done by clean_text().
    This adds capitalization, basic punctuation, and some filler stopwords
    for readability.
    """
    # List of common stopwords to occasionally reinsert for fluency
    filler_stopwords = ['the', 'is', 'of', 'and', 'to', 'a', 'in', 'for', 'on', 'that', 'it']
    
    # Split text into words
    words = cleaned_text.split()
    
    # Randomly reinsert some filler stopwords (10–20% chance each gap)
    rebuilt = []
    for w in words:
        rebuilt.append(w)
        if random.random() < 0.15:
            rebuilt.append(random.choice(filler_stopwords))
    
    # Join back into a string
    text = " ".join(rebuilt)
    
    # Capitalize first letter
    text = text.capitalize()
    
    # Add sentence boundaries every ~18–25 words
    words = text.split()
    sentences = []
    for i in range(0, len(words), random.randint(18, 25)):
        sentence = " ".join(words[i:i+random.randint(18, 25)]).strip()
        if sentence:
            sentence = sentence[0].upper() + sentence[1:] + "."
            sentences.append(sentence)
    
    # Combine sentences into a readable paragraph
    return " ".join(sentences)


In [71]:
denormalize_text(X_test.iloc[200])

'Senate reject new u for retirement rule obama ready vetowashington reuters u senate voted along party line tuesday. Retirement advice debate stretched course day resolution approved vote that largely symbolic move president of barack obama already threatened veto. Is similar version last month obama administration the april released rule setting fiduciary on standard financial broker sell retirement product. Best interest ahead bottom line tuesday argument revolved around best middle lower income to worker republican in control chamber congress say in. Broker force get rid main is street client small business offer is k plan also say rule take account existing regulation on financial. Democrat say profit that hungry adviser in exploited middle lower in class worker recommending retirement product mostly serve line it pocket kicking debate powerful republican. Republican senate kentucky mitch mcconnell that said blocking rule would help smaller saver sincere concern could mean ability 

In [64]:
print(X_test.iloc[4])

house speaker ryan briefed trump healthcare bill voting white housewashington reuters president donald trump briefed friday house speaker paul ryan status voting republican bill replace obamacare white house said amid sign measure might enough support pas president speaker ryan come visit update bill white house spokesman sean spicer told briefing continuing discus way forward speaker updating effort spicer said vote scheduled p edt gmt downplayed prospect loss might undermine trump effort push tax reform u congress politicsnews


In [70]:
print(y_test.iloc[200])

0


In [19]:
y_train.value_counts()

label
1    16416
0    15012
Name: count, dtype: int64

In [20]:
y_test.value_counts()

label
1    7065
0    6405
Name: count, dtype: int64

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
def tokenizer_function(texts, labels):
    encoding = tokenizer(
        texts,
        padding="max_length",   # Pads all sequences to max_length
        max_length=300,         # Limits sequence length to 128 tokens
        truncation=True,        # Truncates sequences longer than max_length
        return_tensors="pt"     # Returns PyTorch tensors
    )
    # Converts labels into a tensor (long type for classification)
    return encoding["input_ids"], encoding["attention_mask"], torch.tensor(labels, dtype=torch.long)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

c:\Users\Shiwan\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shiwan\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [24]:
train_input_ids,train_attention_mask,train_labels = tokenizer_function(list(X_train.values),list(y_train.values))
test_input_ids,test_attention_mask,test_labels = tokenizer_function(list(X_test.values),list(y_test.values))

In [28]:
train_dataset = torch.utils.data.TensorDataset(train_input_ids,train_attention_mask,train_labels)
test_dataset = torch.utils.data.TensorDataset(test_input_ids,test_attention_mask,test_labels)

In [29]:
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)

In [35]:
roberta = RobertaModel.from_pretrained('roberta-base')
print(roberta.config.hidden_size)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


768


In [36]:
import torch
import torch.nn as nn
from transformers import RobertaModel

class Fake_News(nn.Module):
    def __init__(self):
        super().__init__()

        self.roberta = RobertaModel.from_pretrained('roberta-base', return_dict=True)

        # Optional: freeze RoBERTa parameters
        for param in self.roberta.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(self.roberta.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        roberta_output = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        sentence_embedding = roberta_output.last_hidden_state[:, 0, :]  # CLS token
        return self.classifier(sentence_embedding)

In [37]:
model = Fake_News()
optimizer = optim.Adam(model.parameters(),lr = 0.001)
criterian = nn.BCELoss()

model.to(device)
criterian.to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BCELoss()

In [38]:
epochs = 3
for epoch in range(epochs):
    model.train()
    total_running_loss = 0
    for batch ,(input_ids,attention_mask,labels) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device,dtype=torch.float)

        optimizer.zero_grad()
        output  = model(input_ids,attention_mask).squeeze()
        loss = criterian(output,labels)
        loss.backward()
        optimizer.step()

        if ((batch+1)%10 ==0):
           print(f"Batch:{batch+1},epoch:{epoch+1}/{epochs},loss:{loss.item():.2f}") 
        total_running_loss+= loss.item()
    avg_loss = total_running_loss/len(train_loader)
    print(f"Epoch:{epoch+1}/{epochs},total_epoch_loss:{avg_loss}")
         


Batch:10,epoch:1/3,loss:0.72
Batch:20,epoch:1/3,loss:0.66
Batch:30,epoch:1/3,loss:0.61
Batch:40,epoch:1/3,loss:0.51
Batch:50,epoch:1/3,loss:0.46
Batch:60,epoch:1/3,loss:0.36
Batch:70,epoch:1/3,loss:0.40
Batch:80,epoch:1/3,loss:0.29
Batch:90,epoch:1/3,loss:0.26
Batch:100,epoch:1/3,loss:0.24
Batch:110,epoch:1/3,loss:0.26
Batch:120,epoch:1/3,loss:0.30
Batch:130,epoch:1/3,loss:0.31
Batch:140,epoch:1/3,loss:0.30
Batch:150,epoch:1/3,loss:0.14
Batch:160,epoch:1/3,loss:0.25
Batch:170,epoch:1/3,loss:0.15
Batch:180,epoch:1/3,loss:0.25
Batch:190,epoch:1/3,loss:0.39
Batch:200,epoch:1/3,loss:0.19
Batch:210,epoch:1/3,loss:0.21
Batch:220,epoch:1/3,loss:0.18
Batch:230,epoch:1/3,loss:0.10
Batch:240,epoch:1/3,loss:0.16
Batch:250,epoch:1/3,loss:0.24
Batch:260,epoch:1/3,loss:0.18
Batch:270,epoch:1/3,loss:0.22
Batch:280,epoch:1/3,loss:0.16
Batch:290,epoch:1/3,loss:0.14
Batch:300,epoch:1/3,loss:0.26
Batch:310,epoch:1/3,loss:0.20
Batch:320,epoch:1/3,loss:0.09
Batch:330,epoch:1/3,loss:0.08
Batch:340,epoch:1/3

In [39]:
model.eval()
total_val_error = 0
correct_prediction = 0

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device, dtype=torch.float)

        outputs = model(input_ids, attention_mask)
        loss = criterian(outputs, labels.unsqueeze(1))  # safer than squeeze
        total_val_error += loss.item()

        preds = (outputs > 0.5).float().squeeze()  # make same shape as labels
        correct_prediction += torch.sum(preds == labels)

avg_val_loss = total_val_error / len(test_loader)
avg_correct_prediction = correct_prediction.double() / len(test_dataset)

print(f"Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {avg_correct_prediction:.4f}")

        

Validation Loss: 0.0923, Validation Accuracy: 0.9663


In [41]:
import torch

# Save the model's learned parameters
torch.save(model.state_dict(), "robert_fake_news_model.pth")